# **IMDB Bag of Words: Pipeline 1**

This experiment implements a classical NLP baseline using the
**Bag of Words (BoW)** representation over the raw IMDB dataset
without preprocessing transformations.

Pipeline 1 performs the following operations sequentially:

1. Remove special characters and numbers
2. Tokenize the text into words
3. Convert all tokens to lowercase
4. Remove stopwords
5. Apply lemmatization
6. Reconstruct the cleaned text

The objective of this experiment is to:

- Build Bag of Words representations
- Train baseline machine learning classifiers
- Evaluate classification performance
- Compare traditional NLP approaches against future embedding architectures



# **1. Import Libraries**

In [2]:
import pandas as pd
import numpy as np
import os

from sklearn.feature_extraction.text import CountVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

import matplotlib.pyplot as plt
import seaborn as sns
import time

# **2. Load Dataset**

Pipeline 0 uses the raw IMDB dataset without preprocessing.

The dataset is divided into:
- Training set
- Validation set
- Testing set

In [3]:
train = pd.read_csv(
    r"C:\Users\Ale\Documents\Final_NLP\Homework_2\Text-Representation-and-Basic-Text-Mining\Text_Classification_and_Embeddings\Models\Data\preprocessed_data\pipeline_1\train.csv"
)

val = pd.read_csv(
    r"C:\Users\Ale\Documents\Final_NLP\Homework_2\Text-Representation-and-Basic-Text-Mining\Text_Classification_and_Embeddings\Models\Data\preprocessed_data\pipeline_1\validation.csv"
)

test = pd.read_csv(
    r"C:\Users\Ale\Documents\Final_NLP\Homework_2\Text-Representation-and-Basic-Text-Mining\Text_Classification_and_Embeddings\Models\Data\preprocessed_data\pipeline_1\test.csv"
)

print("Train Shape:", train.shape)
print("Validation Shape:", val.shape)
print("Test Shape:", test.shape)

Train Shape: (20000, 2)
Validation Shape: (2500, 2)
Test Shape: (2500, 2)


# **3. Bag of Words Feature Extraction**

Bag of Words converts textual documents into numerical vectors
based on word occurrence frequencies.

Each position in the vector corresponds to a vocabulary term,
and the values represent the frequency of occurrence inside the document.

In [4]:
def compute_bag_of_words(
    X_train,
    X_val,
    X_test,
    max_features=10000
):

    """
    Computes Bag of Words representations
    for train, validation, and test sets.
    """

    vectorizer = CountVectorizer(
        max_features=max_features
    )

    # Fit only on training data
    X_train_bow = vectorizer.fit_transform(X_train)

    # Transform validation and test
    X_val_bow = vectorizer.transform(X_val)
    X_test_bow = vectorizer.transform(X_test)

    return (
        X_train_bow,
        X_val_bow,
        X_test_bow,
        vectorizer
    )

# **4. Prepare Features and Labels**

In [5]:
X_train = train['text']
y_train = train['label']

X_val = val['text']
y_val = val['label']

X_test = test['text']
y_test = test['label']

In [6]:
X_train_bow, X_val_bow, X_test_bow, bow_vectorizer = compute_bag_of_words(
    X_train,
    X_val,
    X_test
)

print("Bag of Words Shapes:")
print("Train:", X_train_bow.shape)
print("Validation:", X_val_bow.shape)
print("Test:", X_test_bow.shape)

Bag of Words Shapes:
Train: (20000, 10000)
Validation: (2500, 10000)
Test: (2500, 10000)


# **5. Model Training**

Two classical machine learning classifiers are trained:

- Logistic Regression
- Multinomial Naive Bayes

These models are commonly used as traditional NLP baselines.

In [7]:
def train_model(model, X_train, y_train):

    """
    Trains a machine learning model.
    """
    model.fit(X_train, y_train)
    return model

In [8]:
# Logistic Regression
logistic_model = LogisticRegression(max_iter=1000)

logistic_model = train_model(
    logistic_model,
    X_train_bow,
    y_train
)

In [9]:
# Naive Bayes
naive_bayes_model = MultinomialNB()

naive_bayes_model = train_model(
    naive_bayes_model,
    X_train_bow,
    y_train
)

# **6. Evaluation Metrics**

The following metrics are computed:

- Accuracy
- Precision
- Recall
- F1-Score

These metrics evaluate classification performance
from different perspectives.

In [10]:

def compute_metrics(model, X_test, y_test):

    """
    Computes classification metrics
    and inference time.
    """

    start_time = time.time()
    # Predictions
    predictions = model.predict(X_test)
    end_time = time.time()
    inference_time = end_time - start_time

    # Average inference time per sample
    avg_inference_time = inference_time / len(y_test)

    # Compute metrics
    metrics = {
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions),
        "Recall": recall_score(y_test, predictions),
        "F1-Score": f1_score(y_test, predictions),
        "Inference_Time_Seconds": inference_time,
        "Average_Inference_Time_Per_Sample":avg_inference_time      
    }

    return metrics, predictions

In [11]:
# Logistic Regression Metrics
log_metrics, log_predictions = compute_metrics(
    logistic_model,
    X_test_bow,
    y_test
)

print("Logistic Regression Metrics")
print(log_metrics)

Logistic Regression Metrics
{'Accuracy': 0.8752, 'Precision': 0.8658346333853354, 'Recall': 0.888, 'F1-Score': 0.8767772511848341, 'Inference_Time_Seconds': 0.0010333061218261719, 'Average_Inference_Time_Per_Sample': 4.1332244873046876e-07}


In [12]:
# Naive Bayes Metrics
nb_metrics, nb_predictions = compute_metrics(
    naive_bayes_model,
    X_test_bow,
    y_test
)

print("\nNaive Bayes Metrics")
print(nb_metrics)


Naive Bayes Metrics
{'Accuracy': 0.8468, 'Precision': 0.8573784006595219, 'Recall': 0.832, 'F1-Score': 0.8444985789687373, 'Inference_Time_Seconds': 0.0009975433349609375, 'Average_Inference_Time_Per_Sample': 3.99017333984375e-07}


# **7. Confusion Matrix**

The confusion matrix allows visualization of:

- True Positives
- True Negatives
- False Positives
- False Negatives

This helps analyze classification behavior and prediction errors.

In [13]:
def plot_confusion_matrix(
    y_true,
    y_pred,
    model_name,
    pipeline_name,
    representation_name,
    output_dir="results_bag"
):

    """
    Computes and saves confusion matrix.

    The confusion matrix includes:
    - Model name
    - Pipeline name
    - Representation type

    Confusion matrices are saved inside:
    conf_matrix/
    """
    # Create confusion matrix directory
    conf_matrix_dir = os.path.join(
        output_dir,
        "conf_matrix"
    )

    os.makedirs(conf_matrix_dir, exist_ok=True)
    cm = confusion_matrix(y_true, y_pred)

    plt.figure(figsize=(6, 5))

    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues'
    )

    plt.title(f"{model_name} | {pipeline_name} | {representation_name}")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")

    # Save path
    file_name = (
        f"{model_name}_"
        f"{pipeline_name}_"
        f"{representation_name}_"
        f"confusion_matrix.png"
    )

    save_path = os.path.join(
        conf_matrix_dir,
        file_name
    )

    # Save figure
    plt.savefig(
        save_path,
        bbox_inches='tight'
    )

    plt.close()

    print(f"Confusion matrix saved in:\n{save_path}")

In [14]:
# Create Results Directory
results_dir = "results_bag"
os.makedirs(results_dir, exist_ok=True)


# Plot Confusion Matrices

plot_confusion_matrix(
    y_true=y_test,
    y_pred=log_predictions,
    model_name="LogisticRegression",
    pipeline_name="pipeline_1",
    representation_name="BagOfWords"
)

plot_confusion_matrix(
    y_true=y_test,
    y_pred=nb_predictions,
    model_name="NaiveBayes",
    pipeline_name="pipeline_1",
    representation_name="BagOfWords"
)

Confusion matrix saved in:
results_bag\conf_matrix\LogisticRegression_pipeline_1_BagOfWords_confusion_matrix.png
Confusion matrix saved in:
results_bag\conf_matrix\NaiveBayes_pipeline_1_BagOfWords_confusion_matrix.png


# **8. Save Results**

All evaluation metrics and reports are saved
inside the `results_bag/` directory for future analysis.

In [15]:
def save_results(
    metrics,
    model_name,
    pipeline_name,
    representation_name,
    output_dir="results_bag",
    results_file="experiment_results.csv"
):

    """
    Saves experiment results into a single CSV file.
    Each experiment is appended as a new row.
    """

    # Create results directory
    os.makedirs(output_dir, exist_ok=True)

    # Full CSV path
    save_path = os.path.join(
        output_dir,
        results_file
    )

    # Create experiment dictionary
    results = {
        "Model": model_name,
        "Pipeline": pipeline_name,
        "Representation": representation_name,
        **metrics
    }

    # Convert current experiment to dataframe
    current_result_df = pd.DataFrame([results])

    # Append if file exists
    if os.path.exists(save_path):
        existing_results = pd.read_csv(save_path)
        updated_results = pd.concat(
            [existing_results, current_result_df],
            ignore_index=True
        )
    else:
        updated_results = current_result_df

    # Save updated dataframe
    updated_results.to_csv(
        save_path,
        index=False
    )

    print(f"Results appended to:\n{save_path}")

In [16]:
# Save Logistic Regression Results
save_results(
    metrics=log_metrics,
    model_name="LogisticRegression",
    pipeline_name="pipeline_1",
    representation_name="BagOfWords"
)

# Save Naive Bayes Results
save_results(
    metrics=nb_metrics,
    model_name="NaiveBayes",
    pipeline_name="pipeline_1",
    representation_name="BagOfWords"
)
    

print("Results saved successfully inside:")
print(results_dir)

Results appended to:
results_bag\experiment_results.csv
Results appended to:
results_bag\experiment_results.csv
Results saved successfully inside:
results_bag
